In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
uber = pd.read_csv('uber.csv')

In [3]:
uber.head()

In [4]:
uber.info()

In [5]:
uber.isnull().sum()

In [6]:
uber_2 = uber.drop(['Unnamed: 0', 'key'], axis=1)
uber_2.dropna(axis=0, inplace=True)

In [7]:
uber_2.isnull().sum()

In [8]:
def haversine(lon_1, lon_2, lat_1, lat_2):
    lon_1, lon_2, lat_1, lat_2 = map(np.radians, [lon_1, lon_2, lat_1, lat_2])
    diff_lon = lon_2 - lon_1
    diff_lat = lat_2 - lat_1
    km = 2 * 6371 * np.arcsin(np.sqrt(np.sin(diff_lat/2.0)**2 + np.cos(lat_1) * np.cos(lat_2) * np.sin(diff_lon/2.0)**2))
    return km

In [9]:
uber_2['Distance'] = haversine(uber_2['pickup_longitude'], uber_2['dropoff_longitude'], uber_2['pickup_latitude'], uber_2['dropoff_latitude'])
uber_2['Distance'] = uber_2['Distance'].astype(float).round(2)

In [10]:
uber_2.head()

In [11]:
plt.scatter(uber_2['Distance'], uber_2['fare_amount'])
plt.xlabel("Distance")
plt.ylabel("fare_amount")

In [12]:
uber_2.drop(uber_2[uber_2['Distance'] > 60].index, inplace = True)
uber_2.drop(uber_2[uber_2['Distance'] == 0].index, inplace = True)
uber_2.drop(uber_2[uber_2['fare_amount'] == 0].index, inplace = True)
uber_2.drop(uber_2[uber_2['fare_amount'] < 0].index, inplace = True)

In [13]:
uber_2.drop(uber_2[(uber_2['fare_amount']>100) & (uber_2['Distance']<1)].index, inplace = True)
uber_2.drop(uber_2[(uber_2['fare_amount']<100) & (uber_2['Distance']>100)].index, inplace = True)

In [14]:
uber_2.info()

In [15]:
plt.scatter(uber_2['Distance'], uber_2['fare_amount'])
plt.xlabel("Distance")
plt.ylabel("fare_amount")

In [16]:
uber_2['pickup_datetime'] = pd.to_datetime(uber_2['pickup_datetime'])
uber_2['Year'] = uber_2['pickup_datetime'].apply(lambda time: time.year)
uber_2['Month'] = uber_2['pickup_datetime'].apply(lambda time: time.month)
uber_2['Day'] = uber_2['pickup_datetime'].apply(lambda time: time.day)
uber_2['Day of Week'] = uber_2['pickup_datetime'].apply(lambda time: time.dayofweek)
uber_2['Day of Week_num'] = uber_2['pickup_datetime'].apply(lambda time: time.dayofweek)
uber_2['Hour'] = uber_2['pickup_datetime'].apply(lambda time: time.hour)
day_map = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
uber_2['Day of Week'] = uber_2['Day of Week'].map(day_map)
uber_2['counter'] = 1

In [17]:
uber_2['pickup'] = uber_2['pickup_latitude'].astype(str) + "," + uber_2['pickup_longitude'].astype(str)
uber_2['drop off'] = uber_2['dropoff_latitude'].astype(str) + "," + uber_2['dropoff_longitude'].astype(str)

In [18]:
uber_2.head()

In [19]:
no_of_trips = []
year = [2009, 2010, 2011, 2012, 2013, 2014, 2015]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
for i in range (2009, 2016):
    x = uber_2.loc [uber_2['Year'] == i, 'counter'].sum()
    no_of_trips.append(x)
print("Average trips a year: ")
print(year, no_of_trips)
plt.bar(year, no_of_trips, color=colors)

In [20]:
no_of_trips = []
month = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
for i in range(1, 13):
    x = uber_2.loc [uber_2['Month'] == i, 'counter'].sum()
    no_of_trips.append(x)
print("Average trips a Month: ")
print(month, no_of_trips)
plt.bar(month, no_of_trips, color=colors)

In [21]:
no_of_trips = []
day = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
for i in range(0, 7):
    x = uber_2.loc[uber_2['Day of Week_num'] == i, 'counter'].sum()
    no_of_trips.append(x)
print("Average trips by Days: ")
print(day, no_of_trips)
plt.bar(day, no_of_trips, color=colors)

In [22]:
year_vs_trips = uber_2.groupby(['Year', 'Month']).agg(
    no_of_trips = ('counter', 'count'),
    Average_fair = ('fare_amount', 'mean'),
    Total_fair = ('fare_amount', 'sum'),
    Avg_distance = ('Distance', 'mean')).reset_index()
year_vs_trips['avg_no_of_trips'] = year_vs_trips['no_of_trips']/30
year_vs_trips['month_year'] = year_vs_trips['Month'].astype(str) +", "+ year_vs_trips['Year'].astype(str)
year_vs_trips = year_vs_trips.reset_index()
year_vs_trips.head()
year_vs_trips.plot(kind='line', x='month_year', y='no_of_trips', xlabel='January, 2009 - June, 2015', ylabel='No of Trips', title='No of trips vs Months')

In [23]:
import seaborn as sns
df_1 = uber_2[['Distance', 'Day of Week_num', 'Hour']].copy()
df_h = df_1.copy()
df_h = df_h.groupby(['Hour', 'Day of Week_num']).mean()
df_h = df_h.unstack(level=0)

In [24]:
fig, ax = plt.subplots(figsize=(24, 7))
sns.heatmap(df_h, cmap="Reds", linewidths=0.3, cbar_kws={"shrink": .8})
xticks_labels = ['12 AM', '01 AM', '02 AM', '03 AM', '04 AM', '05 AM', '06 AM', '07 AM', '08 AM', '09 AM', '10 AM', '11 AM', '12 PM', '01 PM', '02 PM', '03 PM', '04 PM', '05 PM', '06 PM', '07 PM', '08 PM', '09 PM', '10 PM', '11 PM']
yticks_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
plt.xticks(np.arange(24) + .5, labels=xticks_labels)
plt.yticks(np.arange(7) + .5, labels=yticks_labels)
ax.xaxis.tick_top()
title = 'Weekly Uber Rides'.upper()
plt.title(title, fontdict={'fontsize': 25})
plt.show()

In [25]:
import statistics as st
print("Mean of fare prices is %s" % (st.mean(uber_2['fare_amount'])))
print("Median of fare prices is %s" % (st.median(uber_2['fare_amount'])))
print("Standard Deviation of Fare Prices is %s" % (st.stdev(uber_2['fare_amount'])))

In [26]:
import statistics as st
print("Mean of Distance is %s" % (st.mean(uber_2['Distance'])))
print("Median of Distance is %s" % (st.median(uber_2['Distance'])))
print("Standard Deviation of Distance is %s" % (st.stdev(uber_2['Distance'])))

In [27]:
corr = uber_2.corr(numeric_only=True)
corr.style.background_gradient(cmap='BuGn')

In [28]:
X = uber_2['Distance'].values.reshape(-1, 1)
y = uber_2['fare_amount'].values.reshape(-1, 1)

In [29]:
from sklearn.preprocessing import StandardScaler
std = StandardScaler()
y_std = std.fit_transform(y)
print(y_std)
x_std = std.fit_transform(X)
print(x_std)

In [30]:
from sklearn.model_selection import train_test_split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(x_std, y_std, test_size = 0.2, random_state = 0)
from sklearn.linear_model import LinearRegression
l_reg = LinearRegression()
l_reg.fit(X_train, y_train)
print("Training set score: {:.2f}".format(l_reg.score(X_train, y_train)))
print("Test set score: {:.7f}".format(l_reg.score(X_test, y_test)))

In [32]:
y_pred = l_reg.predict(X_test)
df = {'Actual': y_test.flatten(), 'Predicted': y_pred.flatten()}
from tabulate import tabulate
# print(tabulate(df, headers='keys', tablefmt='psql'))

In [33]:
from sklearn import metrics
print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

In [34]:
print(l_reg.intercept_)
print(l_reg.coef_)

In [35]:
plt.subplot(2, 2, 1)
plt.scatter(X_train, y_train, color = 'red')
plt.plot(X_train, l_reg.predict(X_train), color="blue")
plt.title("Fare vs Distance (Training Set)")
plt.ylabel("fare_amount")
plt.xlabel("Distance")
plt.subplot(2,2, 2)
plt.scatter(X_test, y_test, color = 'red')
plt.plot(X_train, l_reg.predict(X_train), color="blue")
plt.ylabel("fare_amount")
plt.xlabel("Distance")
plt.title("Fare vs Distance (Test Set)")
plt.tight_layout()
plt.rcParams["figure.figsize"] = (32,22)
plt.show()